In [1]:
import json
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

DATA = '../data'
pd.set_option('display.width', 200)
print('ready')

ready


In [2]:
start = time.time()
df = pd.read_csv(f'{DATA}/features/all_sessions.csv')
splits = json.load(open(f'{DATA}/splits.json'))
groups = json.load(open(f'{DATA}/features/feature_list.json'))
FEATURES = groups['basic'] + groups['cross_layer']

df = df[df['label_valid'] == 1]
train = df[df['session'].isin(splits['train'])].copy()
print(f'loaded in {time.time() - start:.0f} s')
print('all valid rows :', len(df))
print('training rows  :', len(train), 'from', train['session'].nunique(), 'sessions')
print('features       :', len(FEATURES))

/tmp/ipykernel_112047/3593315750.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{DATA}/features/all_sessions.csv')


loaded in 6 s
all valid rows : 624859
training rows  : 335489 from 40 sessions
features       : 28


In [3]:
summary = train.groupby('event_type').agg(
    rows=('will_degrade', 'size'),
    will_degrade=('will_degrade', 'mean'),
    onset_rows=('onset', 'sum'))
summary['will_degrade'] = (summary['will_degrade'] * 100).round(1)
print(summary)

              rows  will_degrade  onset_rows
event_type                                  
C_BORDER     26144          85.5        3390
C_SEVERE     44829          99.9        3199
F_DEVLINK    17956           6.1         407
F_WAN        16228          96.5        1619
normal      230332           3.8        1411


In [4]:
def score(name, data, pred, prob=None):
    """Scores on protected devices only."""
    keep = (data['role'] == 'protected').values
    y = data['will_degrade'].values[keep]
    p = np.asarray(pred)[keep]
    onset = data['onset'].values[keep] == 1
    quiet = (data['event_type'].values[keep] == 'normal') & (y == 0)
    return {
        'model': name,
        'precision': round(precision_score(y, p), 3),
        'recall': round(recall_score(y, p), 3),
        'f1': round(f1_score(y, p), 3),
        'onset_recall': round(p[onset].mean(), 3),
        'false_alarm_%': round(100 * p[quiet].mean(), 2),
        'pr_auc': round(average_precision_score(y, np.asarray(prob)[keep]), 3) if prob is not None else None,
    }

In [5]:
bad_now = ((train['rtt_ratio'] > 1) | (train['loss_pct'] > 0)).astype(int)
link_full = (train['uplink_util'] > 0.9).astype(int)
results = [score('bad now', train, bad_now), score('link above 90 %', train, link_full)]
print(pd.DataFrame(results).to_string(index=False))

          model  precision  recall    f1  onset_recall  false_alarm_% pr_auc
        bad now      0.971   0.941 0.956         0.496           0.91   None
link above 90 %      0.977   0.718 0.828         0.288           0.48   None


In [6]:
def new_model():
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                          early_stopping=False, random_state=42)

train['prob'] = np.nan
for i, fold in enumerate(splits['folds'], 1):
    start = time.time()
    held = train['session'].isin(fold)
    model = new_model()
    model.fit(train.loc[~held, FEATURES], train.loc[~held, 'will_degrade'])
    train.loc[held, 'prob'] = model.predict_proba(train.loc[held, FEATURES])[:, 1]
    print(f'fold {i}: trained on {(~held).sum()} rows, checked on {held.sum()} rows, {time.time() - start:.0f} s')
train['pred'] = (train['prob'] >= 0.5).astype(int)

fold 1: trained on 268749 rows, checked on 66740 rows, 9 s
fold 2: trained on 268988 rows, checked on 66501 rows, 6 s
fold 3: trained on 268396 rows, checked on 67093 rows, 6 s
fold 4: trained on 267881 rows, checked on 67608 rows, 6 s
fold 5: trained on 267942 rows, checked on 67547 rows, 7 s


In [7]:
results.append(score('trained model', train, train['pred'], train['prob']))
print(pd.DataFrame(results).to_string(index=False))

          model  precision  recall    f1  onset_recall  false_alarm_%  pr_auc
        bad now      0.971   0.941 0.956         0.496           0.91     NaN
link above 90 %      0.977   0.718 0.828         0.288           0.48     NaN
  trained model      0.986   0.943 0.964         0.509           0.28   0.985


In [8]:
prot = train[train['role'] == 'protected']
by_event = prot.groupby('event_type').agg(
    rows=('pred', 'size'),
    truly_positive=('will_degrade', 'mean'),
    alarm_rate=('pred', 'mean'))
by_event['onset_recall'] = prot[prot['onset'] == 1].groupby('event_type')['pred'].mean()
print((by_event * [1, 100, 100, 100]).round(1))

              rows  truly_positive  alarm_rate  onset_recall
event_type                                                  
C_BORDER     18306            85.1        80.4          55.2
C_SEVERE     31381            99.9        96.6          52.9
F_DEVLINK    12567             8.7         7.1          39.5
F_WAN        11354           100.0        95.2          51.4
normal      161253             3.8         3.7          39.9


In [9]:
import os
os.makedirs(f'{DATA}/predictions', exist_ok=True)
cols = ['session', 't_s', 'device', 'role', 'event_type', 'should_act', 'degraded', 'will_degrade', 'onset', 'prob', 'pred']
train[cols].to_csv(f'{DATA}/predictions/train_oof.csv', index=False)
print('saved', len(train), 'rows')

saved 335489 rows
